<a href="https://colab.research.google.com/github/alisony755/DS4400/blob/main/HW3/DS3000_HW3_Problem4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problem 4

In [3]:
# Import libraries
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import pandas as pd
import numpy as np

In [4]:
!pip install -U ucimlrepo

from ucimlrepo import fetch_ucirepo

# Detch dataset
spambase = fetch_ucirepo(id=94)

# Data (as pandas dataframes)
X = spambase.data.features
y = spambase.data.targets

# Print metadata
print(spambase.metadata)

# Print variable information
print(spambase.variables)

{'uci_id': 94, 'name': 'Spambase', 'repository_url': 'https://archive.ics.uci.edu/dataset/94/spambase', 'data_url': 'https://archive.ics.uci.edu/static/public/94/data.csv', 'abstract': 'Classifying Email as Spam or Non-Spam', 'area': 'Computer Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 4601, 'num_features': 57, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': ['Class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1999, 'last_updated': 'Mon Aug 28 2023', 'dataset_doi': '10.24432/C53G6X', 'creators': ['Mark Hopkins', 'Erik Reeber', 'George Forman', 'Jaap Suermondt'], 'intro_paper': None, 'additional_info': {'summary': 'The "spam" concept is diverse: advertisements for products/web sites, make money fast schemes, chain letters, pornography...\n\nThe classification task for this dataset is to determine whether a given email is spam or not.\n\t\nOur collecti

In [5]:
# Convert pandas DataFrames to numpy arrays
X = X.values
y = y.values.ravel()   # flatten to shape (n,)

# Scale features
scaler = StandardScaler()

# Fit scaler on entire dataset
X_scaled = scaler.fit_transform(X)

In [6]:
def k_fold_cv(model, X, y, k):
  """ Performs k-fold cross-validation for a given model.

  Args:
    model: Machine learning model (must have fit() and predict() methods)
    X (numpy.ndarray): Feature matrix of shape (n_samples, n_features)
    y (numpy.ndarray): Target vector of shape (n_samples,)
    k (int): Number of folds

  Returns:
    float: Average validation error across k folds
  """

  # Number of samples in dataset
  n_samples = len(y)

  # Create array of indices (0 to n-1)
  indices = np.arange(n_samples)

  # Shuffle indices to ensure random partitioning
  np.random.seed(42)
  np.random.shuffle(indices)

  # Split indices into k roughly equal folds
  folds = np.array_split(indices, k)

  # List to store validation errors for each fold
  errors = []

  # Loop over each fold
  for i in range(k):

      # Select validation indices (current fold)
      val_idx = folds[i]

      # Select training indices (all other folds)
      train_idx = np.concatenate([folds[j] for j in range(k) if j != i])

      # Create training and validation sets
      X_train, X_val = X[train_idx], X[val_idx]
      y_train, y_val = y[train_idx], y[val_idx]

      # Train model on training set
      model.fit(X_train, y_train)

      # Predict on validation set
      y_pred = model.predict(X_val)

      # Compute validation error (1 - accuracy)
      error = 1 - np.mean(y_pred == y_val)

      # Store error for this fold
      errors.append(error)

  # Compute average validation error across all folds
  avg_error = np.mean(errors)

  return avg_error

In [7]:
# Run CV for logistic regression and LDA

# Values of k to test
k_values = [5, 10]

# Loop through k values
for k in k_values:

    print(f"\n===== k = {k} =====")

    # Logistic Regression model
    log_reg = LogisticRegression(max_iter=5000)

    # Run k-fold CV for logistic regression
    log_error = k_fold_cv(log_reg, X_scaled, y, k)

    print("Logistic Regression Average Validation Error:", log_error)

    # LDA model
    lda = LinearDiscriminantAnalysis()

    # Run k-fold CV for LDA
    lda_error = k_fold_cv(lda, X_scaled, y, k)

    print("LDA Average Validation Error:", lda_error)


===== k = 5 =====
Logistic Regression Average Validation Error: 0.07563470707642923
LDA Average Validation Error: 0.11258296747391779

===== k = 10 =====
Logistic Regression Average Validation Error: 0.07367961897576158
LDA Average Validation Error: 0.11280062246534002
